In [1]:
from pathlib import Path
import hashlib
import json
import re
import shutil
import tarfile
import urllib.parse
import urllib.request
import http.cookiejar
from datetime import datetime, timezone

import pandas as pd

In [ ]:
SHARE_URL = "https://bcmedu-my.sharepoint.com/:f:/g/personal/u249633_bcm_edu/IgCpk9jGz24BSbDiN-YKclOcAUkPg7Bg2jAWWA8TFaT8RSk?e=AcGSgp"

DATA_DIR = Path("Data/official_57_sample_index")
OUT_DIR = Path("Output_files/official_57_sample")

ARCHIVE = DATA_DIR / "chimpanzee_index.tar.gz"
EXTRACT_DIR = DATA_DIR / "extracted"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def open_share(share_url):
    cookies = http.cookiejar.CookieJar()

    opener = urllib.request.build_opener(
        urllib.request.HTTPCookieProcessor(cookies)
    )

    request = urllib.request.Request(
        share_url,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    page = opener.open(
        request,
        timeout=120
    ).read().decode(
        "utf-8",
        errors="replace"
    )

    match = re.search(
        r"var _spPageContextInfo=(\{.*?\});_spPageContextInfo\.updateFormDigestPageLoaded",
        page
    )

    if not match:
        raise RuntimeError(
            "Could not read SharePoint drive information"
        )

    context = json.loads(
        match.group(1)
    )

    drive = context["driveInfo"]

    return (
        opener,
        drive[".driveUrl"],
        drive[".driveAccessToken"]
    )

In [4]:
opener, drive_url, token = open_share(
    SHARE_URL
)

print("SharePoint opened successfully")

SharePoint opened successfully


In [5]:
def find_archive(opener, drive_url, token):

    fields = (
        "id,name,size,file,folder,"
        "@microsoft.graph.downloadUrl"
    )

    query = urllib.parse.urlencode(
        {"$select": fields}
    )

    # List root folder
    root_url = (
        f"{drive_url}/root/children?"
        f"{query}&{token}"
    )

    root = json.load(
        opener.open(root_url)
    )

    # Find chimpanzee folder
    chimp_folders = [
        item
        for item in root.get("value", [])
        if item.get("name") == "chimpanzee"
        and item.get("folder")
    ]

    if len(chimp_folders) != 1:
        raise RuntimeError(
            "Could not uniquely find chimpanzee folder"
        )

    folder_id = urllib.parse.quote(
        str(chimp_folders[0]["id"]),
        safe=""
    )

    # List chimpanzee folder
    child_url = (
        f"{drive_url}/items/{folder_id}/children?"
        f"{query}&{token}"
    )

    items = json.load(
        opener.open(child_url)
    ).get("value", [])

    matches = [
        item
        for item in items
        if item.get("name")
        == "chimpanzee_index.tar.gz"
    ]

    if len(matches) != 1:
        raise RuntimeError(
            "Expected exactly one chimpanzee_index.tar.gz"
        )

    archive = matches[0]

    # If download URL is not directly provided,
    # use the SharePoint content endpoint
    if not archive.get(
        "@microsoft.graph.downloadUrl"
    ):

        item_id = urllib.parse.quote(
            str(archive["id"]),
            safe=""
        )

        archive[
            "@microsoft.graph.downloadUrl"
        ] = (
            f"{drive_url}/items/"
            f"{item_id}/content?{token}"
        )

    return archive

In [10]:
archive_meta = find_archive(
    opener,
    drive_url,
    token
)

archive_meta

In [6]:
def download_file(url, output):

    if output.exists():
        print(
            "Archive already exists:",
            output
        )
        return

    print("Downloading official index...")

    request = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    with urllib.request.urlopen(
        request,
        timeout=300
    ) as response:

        with open(output, "wb") as out:

            shutil.copyfileobj(
                response,
                out,
                length=16 * 1024 * 1024
            )

    print("Download finished")

In [12]:
download_url = archive_meta[
    "@microsoft.graph.downloadUrl"
]

download_file(
    download_url,
    ARCHIVE
)

print(
    "Downloaded size:",
    ARCHIVE.stat().st_size / 1024**3,
    "GiB"
)

Archive already exists: Data/official_57_sample_index/chimpanzee_index.tar.gz
Downloaded size: 6.333972847089171 GiB


In [13]:
remote_size = int(
    archive_meta["size"]
)

local_size = ARCHIVE.stat().st_size

print("Remote bytes:", remote_size)
print("Local bytes :", local_size)

if remote_size != local_size:
    raise RuntimeError(
        "Downloaded file size does not match SharePoint"
    )

print("File size verified")

Remote bytes: 6801051558
Local bytes : 6801051558
File size verified


In [14]:
def sha256sum(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(
                8 * 1024 * 1024
            ),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

In [15]:
archive_sha256 = sha256sum(
    ARCHIVE
)

print(
    "SHA256:",
    archive_sha256
)

SHA256: 4cde68058be46ecfe2aee565df5d032ef56b85f5b9ace2b2c11f18d837c27966


In [16]:
def safe_extract(
    archive,
    destination
):

    destination.mkdir(
        parents=True,
        exist_ok=True
    )

    root = destination.resolve()

    with tarfile.open(
        archive,
        "r:gz"
    ) as tar:

        for member in tar.getmembers():

            target = (
                destination
                / member.name
            ).resolve()

            # Prevent ../ path traversal
            if (
                root not in target.parents
                and target != root
            ):
                raise RuntimeError(
                    f"Unsafe path: {member.name}"
                )

            # Reject links
            if member.issym() or member.islnk():
                raise RuntimeError(
                    f"Links not allowed: {member.name}"
                )

        tar.extractall(
            destination,
            filter="data"
        )

In [17]:
if not EXTRACT_DIR.exists():

    print("Extracting...")

    safe_extract(
        ARCHIVE,
        EXTRACT_DIR
    )

    print("Extraction finished")

else:
    print(
        "Extraction directory already exists"
    )

Extraction directory already exists


In [18]:
def find_index_root(folder):

    candidates = []

    for meta in folder.rglob(
        "meta.txt"
    ):

        parent = meta.parent

        if (
            (parent / "chrom.map").exists()
            and list(
                parent.glob("bptree_*.idx")
            )
        ):
            candidates.append(parent)

    candidates = list(
        set(candidates)
    )

    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one Isopedia index, "
            f"found {len(candidates)}"
        )

    return candidates[0]

In [19]:
index_root = find_index_root(
    EXTRACT_DIR
)

print(
    "Official Isopedia index:",
    index_root
)

Official Isopedia index: Data/official_57_sample_index/extracted/chimpanzee_index


In [20]:
index_files = [
    p
    for p in index_root.rglob("*")
    if p.is_file()
]

print(
    "Index files:",
    len(index_files)
)

Index files: 30


In [21]:
provenance = pd.DataFrame([
    {
        "share_url":
            SHARE_URL,

        "remote_name":
            archive_meta["name"],

        "remote_bytes":
            remote_size,

        "local_archive":
            str(ARCHIVE),

        "local_bytes":
            local_size,

        "sha256":
            archive_sha256,

        "download_verified_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "index_root":
            str(index_root),

        "index_file_count":
            len(index_files)
    }
])

display(provenance)

,share_url,remote_name,remote_bytes,local_archive,local_bytes,sha256,download_verified_utc,index_root,index_file_count
0,https://bcmedu-my.sharepoint.com/:f:/g/persona...,chimpanzee_index.tar.gz,6801051558,Data/official_57_sample_index/chimpanzee_index...,6801051558,4cde68058be46ecfe2aee565df5d032ef56b85f5b9ace2...,2026-08-27T15:36:54.055101+00:00,Data/official_57_sample_index/extracted/chimpa...,30


In [22]:
provenance.to_csv(
    OUT_DIR
    / "index_download_provenance.tsv",
    sep="\t",
    index=False
)